In [1]:
from captum.attr import LayerIntegratedGradients
from config import get_model_config
from models.interpretability import CaseTransformerInterpretability, CaptumWrapper
from models.transformer_helpers import encode_month_year_features, bond_one_hot, discretize_bond_amount
from models.transformer_model import CaseTransformerModel
from pathlib import Path
from transformers import BertModel, BertTokenizer
from typing import Dict, List, Tuple, Optional, Any
from utils.cache_directory_utils import CacheDirectoryManager
from training.teacher_forcing import build_teacher_forced_sequences
from training.top_k_helper import predict_with_top_k_mlb
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import csv
import gc
import matplotlib
matplotlib.use('Agg')  # OPT 3: non-interactive backend
import matplotlib.pyplot as plt
from matplotlib.figure import Figure         
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas 
import numpy as np
import pickle
import random
import re                                      
import string
import sys
import threading                              
import torch
import torch.nn as nn
import pandas as pd
import nltk
import os
import json

In [2]:
try:
    from nltk.corpus import stopwords
    stop_words = frozenset(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    stop_words = frozenset(stopwords.words('english'))

In [3]:
parameters_filename = 'balanced_config_8.json'
PROJECT_ROOT = Path("/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project")
parameters_path = PROJECT_ROOT / "parameters_jsons" / parameters_filename
with open(parameters_path, "r", encoding="utf-8") as file:
    parameters_dict = json.load(file)


In [4]:
class BertToLabelWrapper(nn.Module):
    """
    Wrapper that connects BERT token embeddings directly to label predictions.
    Strictly aligns with CaseTransformerModel feature ordering for interpretability.
    """

    def __init__(
        self,
        bert_model: nn.Module,
        case_transformer: nn.Module,
        target_proceeding_idx: int = -1
    ):
        super().__init__()
        self.bert_model = bert_model
        self.case_transformer = case_transformer
        self.target_proceeding_idx = target_proceeding_idx

        # 1. Capture dynamic dimensions
        self.hidden_dim = case_transformer.hidden_dim

        if hasattr(case_transformer.text_emb, 'out_features'):
            self.bert_out_dim = case_transformer.text_emb.out_features
        elif hasattr(case_transformer.text_emb, 'embedding_dim'):
             self.bert_out_dim = case_transformer.text_emb.embedding_dim
        else:
            self.bert_out_dim = 768

        # 2. Calculate slicing indices based on 'encode_proceeding' order:
        # Order: [Def_Emb, Case_Emb, TEXT_EMB, Mag_Emb, Prior_Lbl, Prior_Chg, Features, Temporal]
        self.pre_text_dim = (self.hidden_dim // 4) + (self.hidden_dim // 4)

    def forward(
        self,
        inputs_embeds: torch.Tensor,            # (batch, seq_len, 768) <- From Captum
        other_features_concat: torch.Tensor,    # (batch, total_unfused_dim) <- Static features
        attention_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Forward pass that:
        1. Contextualizes BERT embeddings
        2. Swaps the static text embedding in 'other_features' with the new BERT output
        3. Runs the full CaseTransformer prediction head
        """
        # --- Step 1: Get contextualized CLS token from BERT ---
        bert_outputs = self.bert_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        # Extract CLS token (index 0)
        # Shape: (batch, bert_out_dim)
        text_embedding = bert_outputs.last_hidden_state[:, 0, :]

        # --- Step 2: Surgical replacement of text features ---
        # Slice A: Everything before text
        pre_text_features = other_features_concat[:, :self.pre_text_dim]

        # Slice B: Everything after text
        # Skip the slice corresponding to the *old* cached text embedding
        post_text_start = self.pre_text_dim + self.bert_out_dim
        post_text_features = other_features_concat[:, post_text_start:]

        # Reassemble: [Pre, NEW_TEXT, Post]
        combined_features = torch.cat(
            [pre_text_features, text_embedding, post_text_features],
            dim=1
        )

        # --- Step 3: CaseTransformer Logic ---
        x = self.case_transformer.fuse(combined_features)
        x = x.unsqueeze(1)  # Batch, Seq_Len=1, Hidden

        # Add Positional Encoding
        seq_idx = self.target_proceeding_idx
        if seq_idx < 0: seq_idx = 0

        # Clamp to max position
        max_pos = self.case_transformer.pos.size(1)
        if seq_idx >= max_pos: seq_idx = max_pos - 1

        pos_vector = self.case_transformer.pos[:, seq_idx : seq_idx+1, :]
        x = x + pos_vector

        # Encoder & Classifier
        encoder_output = self.case_transformer.encoder(x)
        logits = self.case_transformer.classifier(encoder_output[:, -1, :])

        return logits

In [5]:
def load_model_for_captum(
    model_cache_manager: CacheDirectoryManager,
    model_name="final_model",
    device=None,
    verbose=True
):
    model_dir = model_cache_manager.get_cache_dir('exported_models')
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    config_path = model_dir / 'model_config.pkl'
    with open(config_path, 'rb') as f:
        config_dict = pickle.load(f)

    model_config = get_model_config()
    if config_dict['min_year'] != model_config.min_year or config_dict['max_year'] != model_config.max_year:
        model_config.update_date_range(config_dict['min_year'], config_dict['max_year'])

    with open(model_dir / 'charge_vocab.pkl', 'rb') as f:
        charge_vocab = pickle.load(f)
    with open(model_dir / 'mag_vocab.pkl', 'rb') as f:
        mag_vocab = pickle.load(f)
    with open(model_dir / 'label_processor.pkl', 'rb') as f:
        label_processor = pickle.load(f)

    bert_cache_dir = model_cache_manager.get_cache_dir('bert')

    model = CaseTransformerModel(
        charge_vocab=charge_vocab, mag_vocab=mag_vocab,
        bert_cache_dir=bert_cache_dir,
        num_labels=config_dict['num_labels'],
        hidden_dim=config_dict['hidden_dim'],
        max_len=config_dict['max_len'],
        bond_dim=config_dict['bond_dim'],
        temporal_dim=config_dict['temporal_dim'],
        enable_interpretability=True,
    )

    state_dict = torch.load(model_dir / f"{model_name}.pt", map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    if torch.cuda.is_available():
        torch.backends.cudnn.enabled = False

    interpreter = CaseTransformerInterpretability(
        model=model, label_processor=label_processor,
        verbose=verbose, device=device
    )

    return {
        'model': model, 'label_processor': label_processor,
        'interpreter': interpreter, 'device': device,
        'config': config_dict, 'charge_vocab': charge_vocab,
        'mag_vocab': mag_vocab
    }

In [6]:
class BertTokenAttributionAnalyzer:
    """
    Optimized per-word BERT attribution analyzer.

    Major changes vs. original:
      - LIG object created ONCE in __init__, not per attribute call  (OPT 2)
      - Pre-built reference embedding cache by sequence length       (OPT 6)
      - Stopwords computed once at module level                      (OPT 8)
    """

    def __init__(self, model, label_processor, config, bert_model_name="bert-base-uncased",
                 device=None, verbose=True):
        self.model = model
        self.model.eval()
        self.label_processor = label_processor
        self.config = config
        self.label_names = config.get('label_names', getattr(label_processor, 'label_names', []))
        self.num_labels = len(self.label_names)
        self.verbose = verbose
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

        if self.verbose:
            print(f"Loading BERT: {bert_model_name}")
        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
        self.bert_model = BertModel.from_pretrained(bert_model_name).to(self.device)
        self.bert_model.eval()

        self.ref_token_id = self.bert_tokenizer.pad_token_id
        self.sep_token_id = self.bert_tokenizer.sep_token_id
        self.cls_token_id = self.bert_tokenizer.cls_token_id

        self.wrapper = BertToLabelWrapper(self.bert_model, self.model).to(self.device)

        # OPT 2: Create LIG ONCE
        self._lig = LayerIntegratedGradients(self.wrapper, self.bert_model.embeddings)

        # OPT 6: Cache for reference embeddings keyed by seq_len
        self._ref_embed_cache = {}
        self._REF_CACHE_MAX = 64

        if self.verbose:
            print(f"BertTokenAttributionAnalyzer initialized on {self.device}")

    # ── Input construction ──────────────────────────────────────────────────

    def construct_input_ref_pair(self, text):
        text_ids = self.bert_tokenizer.encode(text, add_special_tokens=False, max_length=510, truncation=True)
        input_ids = [self.cls_token_id] + text_ids + [self.sep_token_id]
        ref_input_ids = [self.cls_token_id] + [self.ref_token_id] * len(text_ids) + [self.sep_token_id]
        return (
            torch.tensor([input_ids], device=self.device),
            torch.tensor([ref_input_ids], device=self.device)
        )

    def get_bert_embeddings(self, input_ids):
        return self.bert_model.embeddings(input_ids)

    def _get_ref_embeddings(self, seq_len, ref_input_ids):
        if seq_len not in self._ref_embed_cache:
            with torch.no_grad():
                self._ref_embed_cache[seq_len] = self.get_bert_embeddings(ref_input_ids).detach()
            if len(self._ref_embed_cache) > self._REF_CACHE_MAX:
                oldest_key = next(iter(self._ref_embed_cache))
                del self._ref_embed_cache[oldest_key]
        return self._ref_embed_cache[seq_len]

    # ── Core attribution ────────────────────────────────────────────────────

    def attribute_tokens_for_sequence(
        self, seq, target_label_idx, target_proceeding_idx=-1, n_steps=50
    ):
        if target_proceeding_idx < 0:
            target_proceeding_idx = len(seq) + target_proceeding_idx
        proceeding = seq[target_proceeding_idx]
        text = proceeding.get('text', '') or proceeding.get('description', '')
        if not text.strip():
            raise ValueError(f"Proceeding at index {target_proceeding_idx} has no text")

        self.wrapper.target_proceeding_idx = target_proceeding_idx
        static_features = self.model.encode_proceeding_unfused(proceeding, target_proceeding_idx).to(self.device)

        input_ids, ref_input_ids = self.construct_input_ref_pair(text)
        attention_mask = torch.ones_like(input_ids)

        input_embeddings = self.get_bert_embeddings(input_ids)

        # OPT 6: use cached ref embeddings
        ref_embeddings = self._get_ref_embeddings(input_ids.size(1), ref_input_ids)

        # OPT 2: reuse self._lig
        attributions, delta = self._lig.attribute(
            inputs=input_embeddings,
            baselines=ref_embeddings,
            target=target_label_idx,
            additional_forward_args=(static_features, attention_mask),
            n_steps=n_steps,
            return_convergence_delta=True,
        )

        indices = input_ids[0].detach().tolist()
        tokens = self.bert_tokenizer.convert_ids_to_tokens(indices)
        attributions_sum = attributions.sum(dim=-1).squeeze(0)
        norm = torch.norm(attributions_sum)
        if norm > 0:
            attributions_sum = attributions_sum / norm

        return {
            'tokens': tokens,
            'attributions_sum': attributions_sum.detach().cpu().numpy(),
            'target_label_idx': target_label_idx,
            'target_label': self.label_names[target_label_idx] if self.label_names else str(target_label_idx),
            'convergence_delta': delta.item(),
            'text': text,
            'proceeding_idx': target_proceeding_idx,
        }

    # ── Token merging (unchanged logic) ─────────────────────────────────────

    def _merge_compound_tokens(self, words, scores):
        merged_words, merged_scores = [], []
        i = 0
        while i < len(words):
            w, s = words[i], float(scores[i])

            if w == '$' and i + 1 < len(words) and words[i + 1][0].isdigit():
                j = i + 1
                while j < len(words) and (words[j][0].isdigit() or words[j] in [',', '.']):
                    w += words[j]; s += float(scores[j]); j += 1
                i = j
            elif w[0].isdigit() and i + 1 < len(words) and words[i + 1] in ['/', ':']:
                separator = words[i + 1]; j = i + 1
                while j < len(words):
                    nw = words[j]
                    if nw in ['/', ':']:
                        w += nw; s += float(scores[j]); j += 1
                    elif nw[0].isdigit():
                        w += nw; s += float(scores[j]); j += 1
                    elif nw.lower() in ['am', 'pm'] and separator == ':':
                        w += nw; s += float(scores[j]); j += 1; break
                    else:
                        break
                i = j
            else:
                i += 1

            merged_words.append(w)
            merged_scores.append(s)
        return merged_words, merged_scores

    def aggregate_word_attributions(self, results, remove_stopwords=True, extra_stopwords=None):
        tokens, attrs = results['tokens'], results['attributions_sum']
        SKIP = {'[CLS]', '[SEP]', '[PAD]'}

        words, word_scores = [], []
        current_word, current_score = None, 0.0
        for token, score in zip(tokens, attrs):
            if token in SKIP:
                continue
            if token.startswith('##'):
                if current_word is not None:
                    current_word += token[2:]
                    current_score += float(score)
            else:
                if current_word is not None:
                    words.append(current_word)
                    word_scores.append(current_score)
                current_word, current_score = token, float(score)
        if current_word is not None:
            words.append(current_word)
            word_scores.append(current_score)

        words, word_scores = self._merge_compound_tokens(words, word_scores)

        # OPT 8: use module-level stop_words
        sw = stop_words
        if extra_stopwords:
            sw = sw | frozenset(w.lower() for w in extra_stopwords)

        word_importances, word_counts = {}, {}
        for word, score in zip(words, word_scores):
            key = word.strip(string.punctuation).lower()
            if not key or (remove_stopwords and key in sw):
                continue
            count = word_counts.get(key, 0)
            entry_key = key if count == 0 else f"{key}_{count + 1}"
            word_importances[entry_key] = score
            word_counts[key] = count + 1

        return word_importances

In [7]:
def build_teacher_seq_for_instance(sequences, labels, si, li, label_processor):
    seq_slice = sequences[si][:li + 1]
    labels_slice = labels[si][:li + 1]
    labels_binary = [
        lbl.cpu().numpy() if isinstance(lbl, torch.Tensor) else np.array(lbl, dtype=float)
        for lbl in label_processor.transform_sequence(labels_slice)
    ]
    teacher_seq, _ = build_teacher_forced_sequences(
        seq_slice, labels_binary, label_processor, include_initial_state=True
    )
    return teacher_seq

In [8]:
def find_predicted_instances_split(
    model, sequences, labels, target_labels, label_processor,
    max_per_label=None, seed=42
):
    """
    OPT 1: Batched scanning.
    """
    model.eval()
    predicted_instances = {lbl: {'TP': [], 'FP': []} for lbl in target_labels}
    target_indices = {lbl: label_processor.label_names.index(lbl) for lbl in target_labels}

    SCAN_BATCH_SIZE = 32

    print("🔍 Scanning dataset (batched)...")

    work_items = []
    for si, seq in enumerate(sequences):
        for li in range(1, len(seq)):
            k_labels = len(labels[si][li])
            if k_labels == 0:
                continue
            work_items.append((si, li, k_labels, labels[si][li]))

    for batch_start in range(0, len(work_items), SCAN_BATCH_SIZE):
        batch = work_items[batch_start:batch_start + SCAN_BATCH_SIZE]

        teacher_seqs = []
        for si, li, k_labels, gt in batch:
            teacher_seqs.append(
                build_teacher_seq_for_instance(sequences, labels, si, li, label_processor)
            )

        with torch.no_grad():
            for idx, (si, li, k_labels, gt) in enumerate(batch):
                logits = model(teacher_seqs[idx], return_all_timesteps=True)
                probs = torch.sigmoid(logits).squeeze(0).cpu()
                probability_at_step = probs[-1]
                pred_binary = predict_with_top_k_mlb(probability_at_step, k_labels, label_processor)

                for label_name, lbl_idx in target_indices.items():
                    if int(pred_binary[lbl_idx]) == 1:
                        split = 'TP' if label_name in gt else 'FP'
                        predicted_instances[label_name][split].append((si, li))

        done = min(batch_start + SCAN_BATCH_SIZE, len(work_items))
        if done % (SCAN_BATCH_SIZE * 10) == 0 or done == len(work_items):
            print(f"  Scanned {done}/{len(work_items)} instances...")

    if max_per_label is not None:
        rng = random.Random(seed)
        for label_name in target_labels:
            for split_type in ['TP', 'FP']:
                lst = predicted_instances[label_name][split_type]
                if len(lst) > max_per_label:
                    predicted_instances[label_name][split_type] = rng.sample(lst, max_per_label)

    return predicted_instances

In [9]:
def save_token_chart(results, save_path):
    """
    FIX 3: Use Figure() + FigureCanvas directly instead of plt.subplots().
    This is fully thread-safe because it never touches pyplot's global state.
    """
    tokens = results['tokens']
    attrs = results['attributions_sum']
    label = results['target_label']
    filtered = [(t, a) for t, a in zip(tokens, attrs) if t not in {'[CLS]', '[SEP]', '[PAD]'}]
    if not filtered:
        return
    ftokens, fattrs = zip(*filtered)

    fig = Figure(figsize=(max(15, len(ftokens) * 0.25), 5))
    FigureCanvas(fig)                          # ← attach a canvas so savefig works
    ax = fig.add_subplot(111)
    colors = ['#2ca02c' if a > 0 else '#d62728' for a in fattrs]
    ax.bar(range(len(ftokens)), fattrs, color=colors)
    ax.set_xticks(range(len(ftokens)))
    ax.set_xticklabels(ftokens, rotation=90, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f"Token Attributions — {label}\n"
                 f"seq={results.get('seq_idx', '?')}  "
                 f"step={results.get('label_step', '?')}  "
                 f"proc_idx={results['proceeding_idx']}", fontsize=9)
    ax.set_ylabel("Attribution Score")
    fig.tight_layout()
    fig.savefig(save_path, dpi=100)
    # No plt.close() needed — fig is local and will be GC'd



In [10]:
def save_word_chart(word_importances, label_name, seq_index, label_step, save_path):
    items = list(word_importances.items())
    if not items:
        return
    words, scores = zip(*items)

    fig = Figure(figsize=(max(15, len(words) * 0.3), 5))
    FigureCanvas(fig)
    ax = fig.add_subplot(111)
    colors = ['#2ca02c' if s > 0 else '#d62728' for s in scores]
    ax.bar(range(len(words)), scores, color=colors)
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=90, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f"Word Attributions — {label_name}\nseq={seq_index}  step={label_step}", fontsize=9)
    ax.set_ylabel("Attribution Score")
    fig.tight_layout()
    fig.savefig(save_path, dpi=120)

In [11]:
def save_scores_csv(results, word_importances, save_path):
    tokens = results['tokens']
    attributions = results['attributions_sum']
    word_rows = list(word_importances.items())
    token_rows = [(t, float(a)) for t, a in zip(tokens, attributions) if t not in {'[CLS]', '[SEP]', '[PAD]'}]
    max_len = max(len(token_rows), len(word_rows))
    with open(save_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Token', 'Token Attribution', 'Word', 'Word Attribution'])
        for i in range(max_len):
            t_tok, t_score = token_rows[i] if i < len(token_rows) else ('', '')
            w_word, w_score = word_rows[i] if i < len(word_rows) else ('', '')
            writer.writerow([t_tok, t_score, w_word, w_score])

In [12]:
def _write_instance_files(results, word_imps, label_name, split_type, si, li, prefix):
    save_token_chart(results, str(prefix) + '_tokens.png')
    save_word_chart(word_imps, f"{label_name} {split_type}", si, li, str(prefix) + '_words.png')
    save_scores_csv(results, word_imps, str(prefix) + '_scores.csv')

In [13]:
# Regex to strip _2, _3, etc. suffixes that aggregate_word_attributions adds
_SUFFIX_RE = re.compile(r'_\d+$')

def _base_word_key(word_key: str) -> str:
    """
    'court'   → 'court'
    'court_2' → 'court'
    'court_3' → 'court'
    """
    return _SUFFIX_RE.sub('', word_key)

from dateutil import parser as date_parser

# Month names — written-out dates like "march" or "december"
_MONTH_NAMES = frozenset({
    'january', 'february', 'march', 'april', 'may', 'june',
    'july', 'august', 'september', 'october', 'november', 'december',
    'jan', 'feb', 'mar', 'apr', 'jun', 'jul', 'aug', 'sep', 'sept', 'oct', 'nov', 'dec',
})

def _is_numeric(s: str) -> bool:
    """True if `s` parses as a number after stripping commas and a leading $."""
    s = s.lstrip('$').replace(',', '')
    try:
        float(s)
        return True
    except ValueError:
        return False

def _is_date_money_or_numeric(word: str) -> bool:
    """
    Returns True if `word` looks like a date, time, money amount, year, or
    bare number. Used to filter these out before aggregation.
    """
    if not word:
        return True
    w = word.lower().strip()

    # Month names
    if w in _MONTH_NAMES:
        return True

    # Money / numeric / year — all handled by float() after light cleanup
    if _is_numeric(w):
        return True

    # Dates and times — only invoke dateutil if the token looks date/time-like
    # (contains a separator and at least one digit). This avoids false
    # positives like dateutil parsing "may" or "monday" as a date.
    if any(sep in w for sep in '/:-') and any(c.isdigit() for c in w):
        try:
            date_parser.parse(w, fuzzy=False)
            return True
        except (ValueError, OverflowError):
            pass

    return False


In [ ]:
folder = rf'/data/krosen5/cache/transformers/{parameters_dict["folder_name_directory"]}/'
os.makedirs(folder, exist_ok=True)
model_cache_manager = CacheDirectoryManager(base_path=folder)

components = load_model_for_captum(
    model_cache_manager, model_name="final_model_mlb_best_fold_1", verbose=False
)
model = components['model']
label_processor = components['label_processor']
device = components['device']
label_names = label_processor.label_names

TARGET_LABELS = [
    "PLED_GUILTY_TO_CHARGES",
    "PLED_NOT_GUILTY_TO_CHARGES",
    "TRIAL_VERDICT_REACHED",
    "SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED",
    "SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED",
    "CONTINUANCE_OCCURRED",
    'CHARGES_ACCEPTED',
    'CHARGES_REFUSED'
]

TARGET_LABELS = [
    "SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED"
]
MAX_INSTANCES_PER_LABEL = None
MIN_WORDS = 5
NUMBER_OF_INTEGRATED_GRADIENTS_STEPS = 50
TOP_N_WORDS_TO_DISPLAY = 30
FLUSH_EVERY = 20
IO_WORKERS = 4
MIN_INSTANCE_COUNT = 3

Date range updated: 2008 to 2025


In [15]:
sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
with open(sequences_path, 'rb') as f:
    sequences, labels = pickle.load(f)
print(f"✅ Loaded {len(sequences)} sequences.")


✅ Loaded 2533 sequences.


In [16]:
# ── Scan for predictions ────────────────────────────────────────────────
all_instances = find_predicted_instances_split(
    model=model, sequences=sequences, labels=labels,
    target_labels=TARGET_LABELS, label_processor=label_processor,
    max_per_label=MAX_INSTANCES_PER_LABEL,
)

print("\n📊 Instance Breakdown:")
for label, splits in all_instances.items():
    print(f"  {label}: TP={len(splits['TP'])}  FP={len(splits['FP'])}")

🔍 Scanning dataset (batched)...
  Scanned 320/16926 instances...
  Scanned 640/16926 instances...
  Scanned 960/16926 instances...
  Scanned 1280/16926 instances...
  Scanned 1600/16926 instances...
  Scanned 1920/16926 instances...
  Scanned 2240/16926 instances...
  Scanned 2560/16926 instances...
  Scanned 2880/16926 instances...
  Scanned 3200/16926 instances...
  Scanned 3520/16926 instances...
  Scanned 3840/16926 instances...
  Scanned 4160/16926 instances...
  Scanned 4480/16926 instances...
  Scanned 4800/16926 instances...
  Scanned 5120/16926 instances...
  Scanned 5440/16926 instances...
  Scanned 5760/16926 instances...
  Scanned 6080/16926 instances...
  Scanned 6400/16926 instances...
  Scanned 6720/16926 instances...
  Scanned 7040/16926 instances...
  Scanned 7360/16926 instances...
  Scanned 7680/16926 instances...
  Scanned 8000/16926 instances...
  Scanned 8320/16926 instances...
  Scanned 8640/16926 instances...
  Scanned 8960/16926 instances...
  Scanned 9280/1692

In [17]:
OUTPUT_ROOT = Path(model_cache_manager.get_cache_dir('interpretability')) / 'captum_attributions'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [18]:
analyzer = BertTokenAttributionAnalyzer(
    model=model, label_processor=label_processor,
    config=components['config'], device=device, verbose=False,
)

label_idx_map = {name: idx for idx, name in enumerate(label_names)}
label_word_scores = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

In [19]:
import traceback

In [20]:
io_pool = ThreadPoolExecutor(max_workers=IO_WORKERS)
io_futures = []

for label_name, splits in all_instances.items():
    label_idx = label_idx_map[label_name]

    for split_type, instances in splits.items():
        n = len(instances)
        if n == 0:
            continue

        split_folder = OUTPUT_ROOT / label_name / split_type
        split_folder.mkdir(parents=True, exist_ok=True)

        print(f"\n{'=' * 60}")
        print(f"{label_name} | {split_type} — {n} instances → {split_folder}")
        print(f"{'=' * 60}")

        skipped = 0

        for i, (si, li) in enumerate(instances):
            # ══════════════════════════════════════════════════════════
            # FIX 1: Was `sequences[si][li - 1]` — wrong proceeding!
            #         The attribution targets the LAST element of the
            #         teacher seq built from sequences[si][:li+1],
            #         which is sequences[si][li], NOT [li-1].
            # ══════════════════════════════════════════════════════════
            raw_proc = sequences[si][li]  # ← FIXED (was li - 1)
            text = raw_proc.get('text') or raw_proc.get('description', '')
            if not text or len(text.split()) < MIN_WORDS:
                skipped += 1
                continue

            prefix = split_folder / f"instance_{i:03d}_seq{si}_step{li:02d}"

            try:
                teacher_seq = build_teacher_seq_for_instance(
                    sequences, labels, si, li, label_processor
                )
                results = analyzer.attribute_tokens_for_sequence(
                    seq=teacher_seq,
                    target_label_idx=label_idx,
                    target_proceeding_idx=-1,
                    n_steps=NUMBER_OF_INTEGRATED_GRADIENTS_STEPS,
                )
                results['seq_idx'] = si
                results['label_step'] = li

                word_imps = analyzer.aggregate_word_attributions(results, remove_stopwords=True)
                label_bucket = label_word_scores[label_name]
                for word_key, score in word_imps.items():
                    base_key = _base_word_key(word_key)
                    if _is_date_money_or_numeric(base_key):   # ← NEW filter
                        continue
                    wb = label_bucket[base_key]
                    wb[split_type].append(score)
                    wb['ALL'].append(score)

                # OPT 4: submit file writes to background threads
                results_copy = {
                    'tokens': results['tokens'],
                    'attributions_sum': results['attributions_sum'].copy(),
                    'target_label': results['target_label'],
                    'seq_idx': si, 'label_step': li,
                    'proceeding_idx': results['proceeding_idx'],
                }
                word_imps_copy = dict(word_imps)
                fut = io_pool.submit(
                    _write_instance_files,
                    results_copy, word_imps_copy,
                    label_name, split_type, si, li, prefix,
                )
                io_futures.append(fut)

            except Exception as e:
                skipped += 1
                if 'memory' in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
                    gc.collect()
                    torch.cuda.empty_cache()
                    print(f"  ⚠️  OOM at ({si}, {li}) — cache cleared")
                else:
                    print(f"  ⚠️  Skipping ({si}, {li}): {e}")

            # OPT 5: GC only at intervals
            if (i + 1) % FLUSH_EVERY == 0:
                gc.collect()
                print(f"  [{i + 1}/{n}] done  (skipped: {skipped})")

        print(f"  ✅ {label_name}/{split_type}: {n - skipped} saved, {skipped} skipped")

# Wait for all file I/O to finish
print("\n⏳ Waiting for file writes to complete...")
for fut in io_futures:
    fut.result()  # raises if a write failed
io_pool.shutdown(wait=True)
print(f"✅ All {len(io_futures)} file writes complete.")


SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED | TP — 290 instances → /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions/SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED/TP
  [20/290] done  (skipped: 0)
  [40/290] done  (skipped: 0)
  [60/290] done  (skipped: 0)
  [80/290] done  (skipped: 0)
  [100/290] done  (skipped: 0)
  [120/290] done  (skipped: 0)
  [140/290] done  (skipped: 0)
  [160/290] done  (skipped: 0)
  [180/290] done  (skipped: 0)
  [200/290] done  (skipped: 0)
  [220/290] done  (skipped: 0)
  [240/290] done  (skipped: 0)
  ⚠️  OOM at (1088, 8) — cache cleared
  [260/290] done  (skipped: 1)
  [280/290] done  (skipped: 1)
  ✅ SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED/TP: 289 saved, 1 skipped

SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED | FP — 91 instances → /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions/SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED/FP
  [20/91] done  (skipp

In [21]:
label_top_words = defaultdict(dict)

for label_name in TARGET_LABELS:
    for split_type in ['ALL', 'TP', 'FP']:
        # Build {word: list_of_scores} for this label/split
        word_score_lists = {}
        for word, split_dict in label_word_scores[label_name].items():
            scores_list = split_dict.get(split_type, [])
            if len(scores_list) >= MIN_INSTANCE_COUNT:   # ← reliability filter
                word_score_lists[word] = scores_list

        if not word_score_lists:
            print(f"\n⚠️  No words meet MIN_INSTANCE_COUNT={MIN_INSTANCE_COUNT} "
                  f"for {label_name}/{split_type}")
            continue

        # Compute mean score AND keep count for display
        mean_scores = {w: (np.mean(s), len(s)) for w, s in word_score_lists.items()}
        # Sort by absolute value of mean
        sorted_words = sorted(mean_scores.items(), key=lambda x: abs(x[1][0]), reverse=True)
        label_top_words[label_name][split_type] = sorted_words

        # Instance count for title
        if split_type == 'ALL':
            n_inst = (len(all_instances.get(label_name, {}).get('TP', []))
                        + len(all_instances.get(label_name, {}).get('FP', [])))
        else:
            n_inst = len(all_instances.get(label_name, {}).get(split_type, []))

        # Terminal output
        print(f"\n{'=' * 70}")
        print(f"🔤 Top {TOP_N_WORDS_TO_DISPLAY} words for '{label_name}' | "
              f"{split_type} (n={n_inst}, min_count={MIN_INSTANCE_COUNT}):")
        print(f"{'=' * 70}")
        for word, (score, count) in sorted_words[:20]:
            bar = '█' * int(abs(score) * 50)
            sign = '+' if score > 0 else '-'
            print(f"  {sign}{abs(score):.4f}  (n={count:3d})  {bar}  {word}")

        # Chart — annotate bars with occurrence count so reader sees support
        words_only = [w for w, _ in sorted_words[:TOP_N_WORDS_TO_DISPLAY]]
        scores_only = [s for _, (s, _) in sorted_words[:TOP_N_WORDS_TO_DISPLAY]]
        counts_only = [c for _, (_, c) in sorted_words[:TOP_N_WORDS_TO_DISPLAY]]

        fig, ax = plt.subplots(figsize=(16, 5.5))
        colors = ['#2ca02c' if s > 0 else '#d62728' for s in scores_only]
        bars = ax.bar(range(len(words_only)), scores_only, color=colors)
        ax.set_xticks(range(len(words_only)))
        # Show "word (n=count)" on x-axis so support is visible at a glance
        labels_with_count = [f"{w}  (n={c})" for w, c in zip(words_only, counts_only)]
        ax.set_xticklabels(labels_with_count, rotation=90, ha='right', fontsize=9)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_title(
            f"Mean Word Attributions — {label_name} | {split_type}  "
            f"(n_instances={n_inst}, min_count={MIN_INSTANCE_COUNT})",
            fontsize=11
        )
        ax.set_ylabel("Mean Attribution Score")
        plt.tight_layout()

        agg_folder = OUTPUT_ROOT / label_name
        agg_folder.mkdir(parents=True, exist_ok=True)
        agg_path = agg_folder / f"_AGGREGATE_{split_type}_top{TOP_N_WORDS_TO_DISPLAY}_words.png"
        plt.savefig(agg_path, dpi=150)
        plt.close(fig)
        print(f"Saved: {agg_path}")


🔤 Top 30 words for 'SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED' | ALL (n=381, min_count=3):
  +0.3380  (n= 72)  ████████████████  pdojl
  +0.3256  (n=  3)  ████████████████  bankers
  +0.2780  (n=  3)  █████████████  depositor
  +0.2720  (n=  3)  █████████████  whaley
  +0.2661  (n=  3)  █████████████  bixby
  +0.2563  (n=  4)  ████████████  saleem
  +0.2509  (n=  4)  ████████████  garvey
  +0.2329  (n=  3)  ███████████  art
  +0.2183  (n= 14)  ██████████  negotiations
  +0.2166  (n=  3)  ██████████  boshea
  +0.2065  (n=  3)  ██████████  result
  +0.2014  (n=  4)  ██████████  sentencing
  +0.1990  (n=  4)  █████████  belfield
  +0.1943  (n=  3)  █████████  melvin
  +0.1943  (n= 12)  █████████  subpoena
  +0.1905  (n=  7)  █████████  rule
  +0.1882  (n=  3)  █████████  mikkelsen
  +0.1854  (n=  5)  █████████  diquan
  +0.1831  (n=  3)  █████████  14:95
  +0.1720  (n=  3)  ████████  netterville
Saved: /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attribu

In [22]:
output_path = OUTPUT_ROOT / 'captum_word_attributions.pkl'
save_payload = {
    'label_top_words': dict(label_top_words),
    'label_word_scores': {
        k: {sk: dict(sv) for sk, sv in v.items()}
        for k, v in label_word_scores.items()
    },
    'all_instances': all_instances,
    'target_labels': TARGET_LABELS,
    'output_root': str(OUTPUT_ROOT),
    'config': {
        'max_instances_per_label': MAX_INSTANCES_PER_LABEL,
        'min_words': MIN_WORDS,
        'number_of_integrated_gradients_steps': NUMBER_OF_INTEGRATED_GRADIENTS_STEPS,
        'top_n_words_to_display': TOP_N_WORDS_TO_DISPLAY,
        'min_instance_count': MIN_INSTANCE_COUNT,
    },
}
with open(output_path, 'wb') as f:
    pickle.dump(save_payload, f)
print(f"\n✅ Final payload saved to {output_path}")


✅ Final payload saved to /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions/captum_word_attributions.pkl
